In [2]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [7]:
dados = pd.read_csv("../data/raw/california_housing.csv")
X = dados.drop(columns="MedHouseVal")
y = dados["MedHouseVal"]

# 1ª divisão: separa o teste – igual à Etapa 2
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

# 2ª divisão: tira a validação de dentro do treino
X_treino, X_val, y_treino, y_val = train_test_split(X_treino, y_treino, test_size=0.25, random_state=42)

print(X_treino.shape, X_val.shape, X_teste.shape)

(12384, 8) (4128, 8) (4128, 8)


In [10]:
def avaliar(nome, modelo, X_tr, X_v):
    modelo.fit(X_tr, y_treino)
    previsto = modelo.predict(X_v)
    return {
        "modelo": nome,
        "MAE": mean_absolute_error(y_val, previsto),
        "RMSE": np.sqrt(mean_squared_error(y_val, previsto)),
        "R2": r2_score(y_val, previsto),
    }

In [12]:
resultados = [
    avaliar("Baseline (média)", DummyRegressor(strategy="mean"), X_treino, X_val),
    avaliar("Só MedInc", make_pipeline(StandardScaler(), LinearRegression()),
            X_treino[["MedInc"]], X_val[["MedInc"]]),
    avaliar("Todas as colunas", make_pipeline(StandardScaler(), LinearRegression()),
            X_treino, X_val),
]
pd.DataFrame(resultados).round(3)

,modelo,MAE,RMSE,R2
0,Baseline (média),0.925,1.172,-0.000
1,Só MedInc,0.631,0.843,0.483
2,Todas as colunas,0.533,0.728,0.614


In [13]:
modelo = make_pipeline(StandardScaler(), LinearRegression())
modelo.fit(X_treino, y_treino)

coefs = pd.Series(modelo[-1].coef_, index=X.columns)
coefs.sort_values(key=abs, ascending=False).round(3)

Latitude     -0.888
Longitude    -0.862
MedInc        0.860
AveBedrms     0.312
AveRooms     -0.280
HouseAge      0.120
AveOccup     -0.026
Population   -0.010
dtype: float64